# cfg_scale sensitivity for SD3-Turbo motion pipe

SD3-Turbo is distilled; cfg > 1 sometimes hurts. sweep and see.

In [ ]:
import numpy as np, os, imageio, pandas as pd
from src.inference.t2v import run as run_single
from src.eval.clipsim_temporal import CLIPTemporal
SCALES = [0.0, 1.0, 2.5, 5.0, 7.5, 10.0, 12.5]
PROMPTS = ['a fox running through leaves', 'astronaut planting flag', 'coffee slow motion pour', 'city rainy neon']
os.makedirs('runs/cfg_sweep', exist_ok=True)


In [ ]:
for i, p in enumerate(PROMPTS):
    for c in SCALES:
        run_single(prompt=p, out_path=f'runs/cfg_sweep/p{i}_c{c}.mp4', cfg_scale=c, seed=42)


In [ ]:
clip = CLIPTemporal()
rows = []
for c in SCALES:
    cts = []
    for i, p in enumerate(PROMPTS):
        f = np.stack(list(imageio.get_reader(f'runs/cfg_sweep/p{i}_c{c}.mp4')))
        cts.append(clip.score(p, f)['mean'])
    rows.append({'cfg_scale': c, 'clipT': np.mean(cts)})
pd.DataFrame(rows)
